# HARP-WDN: retrain the PressureGAT models (Net3 + L-Town)

This notebook generates WNTR training data, trains one `PressureGAT` per network, evaluates it on held-out
scenarios, checks that HARP behaves the same with the model's pressures as with the true pressures, and saves the
`.pt` checkpoints to Google Drive.

**Runtime:** pick **GPU (T4 is plenty)** under *Runtime → Change runtime type*. Data generation runs on the CPU
cores (WNTR is CPU-only) and training uses the GPU. Use **High-RAM** if you raise the L-Town scenario count.

**Rough time on a T4 / 2 CPU Colab:** Net3 data ~5 min, L-Town data ~35 min (about 17 min with 4+ cores),
training ~5-15 min each.

**Run cells top to bottom.** Everything slow is cached on Drive, so after a disconnect you can re-run from the top and it
resumes (data tarballs and finished checkpoints are reused).

> What counts as "good enough" is checked in the verdict cell. Those thresholds are targets I chose so that model error
> sits below HARP's 1 m safety margin. They are not a guarantee that training will reach them: read the numbers.

## 1. Configuration

In [ ]:
SOURCE   = "github"      # "github" (public repo, latest main) | "drive_zip" | "existing"
REPO_URL = "https://github.com/nush1729/Optimised-Water-Distribution-System"
BRANCH   = "main"
# "drive_zip": path to a zip of the repo on Drive, made on your Mac with:
#   cd Optimised-Water-Distribution-System && zip -r harp.zip . -x "frontend/node_modules/*" "frontend/dist/*" ".git/*"
ZIP_ON_DRIVE = "/content/drive/MyDrive/harp_wdn/harp.zip"
# "existing": use a repo that is already on disk (e.g. a local run)
EXISTING_REPO = None

DRIVE_DIR = "/content/drive/MyDrive/harp_wdn"      # checkpoints, metrics and data tarballs are kept here
WORK_DIR  = "/content/work"                        # scratch space, wiped on disconnect

# Per-network settings. Scenarios are seeded individually, so results are reproducible.
NETWORKS = {
    "Net3":   dict(n_scenarios=3000, epochs=200, patience=25, batch_size=32, lr=1e-3, residual=False),
    "L-Town": dict(n_scenarios=2000, epochs=150, patience=20, batch_size=16, lr=1e-3, residual=False),
}
# residual=True trains the GAT on the CHANGE from the intact network's pressure instead of
# absolute pressure. Compare both with the evaluation cell and keep whichever is better.

# Bump this whenever the data generator changes: cached tarballs from an older version are ignored.
# v2 = demand noise is really applied (WNTR's base_demand is read-only; v1 silently skipped it).
DATA_VERSION = "v2"
DOWNLOAD_MODELS = False     # True: also download the .pt files to your computer at the end

# Targets used by the verdict cell (see note at the top).
TARGETS = dict(MAE_m=1.0, R2=0.90, gate_decision_agreement=0.97, gate_false_servable=0.02, harp_cds_gap=0.02)
RUN_TESTS = True

## 2. Environment (Drive, GPU)

In [ ]:
import os, sys, shutil, subprocess, time, json, glob

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Not running on Colab: Drive steps are skipped, DRIVE_DIR is just a local folder.")
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU - training will be slow")
except ImportError:
    print("torch not installed yet (next cells install it)")
print("CPU cores:", os.cpu_count())

## 3. Get the code

In [ ]:
REPO_DIR = os.path.join(WORK_DIR, "repo")

if SOURCE == "existing":
    REPO_DIR = EXISTING_REPO
elif SOURCE == "github":
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
    print("NOTE: GitHub only has what you have pushed; use SOURCE='drive_zip' to test local, unpushed changes.")
elif SOURCE == "drive_zip":
    # Look in the configured path first, then the usual spots, so a zip dropped in My Drive's root still works.
    candidates = [ZIP_ON_DRIVE, "/content/drive/MyDrive/harp.zip", os.path.join(DRIVE_DIR, "harp.zip")]
    zip_path = next((c for c in candidates if os.path.exists(c)), None)
    assert zip_path, "harp.zip not found. Looked in:\n  " + "\n  ".join(candidates)
    print("using", zip_path)
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run(["unzip", "-q", zip_path, "-d", REPO_DIR], check=True)
else:
    raise ValueError(SOURCE)

assert os.path.exists(os.path.join(REPO_DIR, "backend", "gat", "train.py")), "repo layout not found"
# Guard against a stale checkout missing the changes this notebook relies on.
gen_src = open(os.path.join(REPO_DIR, "backend", "gat", "dataset_generator.py")).read()
assert "workers" in gen_src and "_write_split" in gen_src, \
    "dataset_generator.py is the old version (no parallel generation). Push/zip the updated repo."
cfg_src = open(os.path.join(REPO_DIR, "backend", "config.py")).read()
assert "HARP_GENERATED_DIR" in cfg_src, "config.py is the old version (no env-var output dirs)."
print("repo:", REPO_DIR)

## 4. Install dependencies

In [ ]:
# torch is preinstalled on Colab. wntr>=1.5 needs numpy>=2.2.6, which torch>=2.3 supports.
pkgs = ["numpy>=2.2.6", "wntr>=1.5.0", "networkx", "pandas", "scipy", "scikit-learn", "matplotlib", "fastapi", "httpx", "pytest"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
try:
    import torch_geometric  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"], check=True)
print("installed")

> If the next cell fails with a numpy ABI error ("compiled using NumPy 1.x cannot be run in NumPy 2.x"), use
> *Runtime → Restart session* and run from the top. That is the standard fix after upgrading numpy.

## 5. Point the project at Drive and sanity-check

In [ ]:
GEN_DIR     = os.path.join(WORK_DIR, "generated")       # pickles: local disk, fast
MODELS_DIR  = os.path.join(DRIVE_DIR, "trained_models") # checkpoints: on Drive, survive disconnects
RESULTS_DIR = os.path.join(DRIVE_DIR, "results")
for d in (GEN_DIR, MODELS_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

# Must be set BEFORE importing backend.* (config reads them at import time; worker processes inherit them).
os.environ["HARP_GENERATED_DIR"] = GEN_DIR
os.environ["HARP_MODELS_DIR"]    = MODELS_DIR
os.environ["HARP_RESULTS_DIR"]   = RESULTS_DIR
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import warnings; warnings.filterwarnings("ignore")
import logging; logging.getLogger("wntr").setLevel(logging.ERROR)

if RUN_TESTS:
    r = subprocess.run([sys.executable, "-m", "pytest", "backend/tests", "-q", "-p", "no:cacheprovider", "-x"],
                       capture_output=True, text=True, cwd=REPO_DIR,
                       env={k: v for k, v in os.environ.items() if not k.startswith("HARP_")})  # tests use the repo defaults
    print(r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-500:])
    assert r.returncode == 0, "tests failed, fix before training:\n" + r.stdout[-1500:]

## 6. Generate training data (CPU, parallel, cached on Drive)

Each scenario is one random single-pipe failure with log-normal demand noise, simulated for 24 h in WNTR; the label is
the hour-12 pressure at every node. Scenarios that do not converge, or give physically implausible pressures, are skipped
(about 10-15% on Net3).

In [ ]:
import tarfile
from backend.gat.dataset_generator import generate_scenarios

def ensure_data(net, n):
    tar_path = os.path.join(DRIVE_DIR, f"data_{net}_{n}_{DATA_VERSION}.tar")
    split_path = os.path.join(GEN_DIR, f"{net}_split.json")
    if os.path.exists(tar_path):
        print(f"[{net}] restoring cached data from Drive ...")
        with tarfile.open(tar_path) as t:
            t.extractall(GEN_DIR)
        return
    t0 = time.time()
    workers = max(1, (os.cpu_count() or 2))
    print(f"[{net}] generating {n} scenarios with {workers} worker(s) ...")
    stats = generate_scenarios(net, n, workers=workers)
    print(f"[{net}] {stats} in {(time.time()-t0)/60:.1f} min")
    with tarfile.open(tar_path, "w") as t:      # uncompressed: pickles of floats do not compress well
        for f in sorted(glob.glob(os.path.join(GEN_DIR, f"{net}_*.pkl"))) + [split_path]:
            t.add(f, arcname=os.path.basename(f))
    print(f"[{net}] cached to {tar_path}")

for net, cfg in NETWORKS.items():
    ensure_data(net, cfg["n_scenarios"])

## 7. Train

Uses the repo's `train_model` (weighted sampling toward high-impact failures, physics regulariser, early stopping on
validation loss, `ReduceLROnPlateau` learning-rate schedule). It trains on the GPU and then evaluates the best checkpoint on CPU.
Finished checkpoints on Drive are reused unless you set `FORCE_RETRAIN = True`.

In [ ]:
import torch
from backend.gat.train import train_model

FORCE_RETRAIN = False
for net, cfg in NETWORKS.items():
    ckpt = os.path.join(MODELS_DIR, f"{net}_gat.pt")
    if os.path.exists(ckpt) and not FORCE_RETRAIN:
        print(f"[{net}] checkpoint already on Drive, skipping (set FORCE_RETRAIN=True to redo)")
        continue
    print(f"\n===== Training {net} =====")
    t0 = time.time()
    train_model(net, max_epochs=cfg["epochs"], patience=cfg["patience"],
                batch_size=cfg["batch_size"], lr=cfg["lr"], residual=cfg.get("residual", False))
    print(f"[{net}] trained in {(time.time()-t0)/60:.1f} min")

### Training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(NETWORKS), figsize=(6 * len(NETWORKS), 3.5), squeeze=False)
for ax, net in zip(axes[0], NETWORKS):
    p = os.path.join(RESULTS_DIR, f"training_{net}_log.csv")
    if not os.path.exists(p):
        ax.set_title(f"{net}: no log"); continue
    df = pd.read_csv(p)
    ax.plot(df["epoch"], df["train_loss"], label="train")
    ax.plot(df["epoch"], df["val_loss"], label="val")
    ax.set_yscale("log"); ax.set_title(net); ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout(); plt.show()

## 8. Evaluate on held-out scenarios

For each network this reports:
- error vs a naive "predict the mean" model, overall and on the high-impact quartile (failures that really move pressure);
- how often the model makes the same servable / not-servable call as the true pressures at demand nodes. This is the
  decision HARP's hydraulic gate depends on;
- whether MC-Dropout uncertainty tracks real error;
- HARP end-to-end: delivered demand using GAT pressures vs using the true WNTR pressures.

In [ ]:
from backend.gat.evaluate import evaluate_model, harp_end_to_end

rows = []
for net in NETWORKS:
    print(f"--- {net} ---")
    m = evaluate_model(net)
    h = harp_end_to_end(net, n_scenarios=20)
    rows.append({**m,
                 "CDS_no_recovery": h["CDS_no_recovery"],
                 "CDS_harp_true_pressure": h["CDS_harp_true_pressure"],
                 "CDS_harp_gat_pressure": h["CDS_harp_gat_pressure"],
                 "harp_cds_gap": h["max_abs_gap_gat_vs_true"]})

report = pd.DataFrame(rows).set_index("network").T
report.to_csv(os.path.join(RESULTS_DIR, "gat_retrain_report.csv"))
report

### Verdict against the targets

In [ ]:
def verdict(r):
    checks = {
        f"MAE < {TARGETS['MAE_m']} m":                    r["MAE_m"] < TARGETS["MAE_m"],
        f"R2 > {TARGETS['R2']}":                          r["R2"] > TARGETS["R2"],
        f"gate agreement > {TARGETS['gate_decision_agreement']}": r["gate_decision_agreement"] > TARGETS["gate_decision_agreement"],
        f"gate false-servable < {TARGETS['gate_false_servable']}": r["gate_false_servable"] < TARGETS["gate_false_servable"],
        f"HARP CDS gap (GAT vs true) < {TARGETS['harp_cds_gap']}": r["harp_cds_gap"] < TARGETS["harp_cds_gap"],
    }
    return checks

for r in rows:
    print(f"\n== {r['network']} ==")
    for name, ok in verdict(r).items():
        print(("  PASS  " if ok else "  FAIL  ") + name)
    print(f"  (MC-Dropout std vs error, Spearman: {r['mc_dropout_spearman_std_vs_error']:.2f}; "
          "below ~0.3 means do not present the uncertainty as a calibrated confidence)")

### If a check fails

- **MAE far above 1 m on Net3/L-Town:** raise `n_scenarios` (Net3 to 6000, L-Town to 4000) and `epochs`, delete the
  `data_*.tar` (and bump `DATA_VERSION` if you changed the generator) and the checkpoint on Drive (or set `FORCE_RETRAIN = True`) and re-run. More data helps most here
  because a single failed pipe changes only a few edges of the graph.
- **Low gate agreement but fine MAE:** the error is concentrated near the 8 m service threshold. On Net3 almost every demand node sits far above it,
  so this mostly matters for stressed networks.
- **Good numbers, but HARP CDS gap is large:** the problem is in HARP, not the model. Do not keep retraining.
- Report whatever you get. Do not edit the thresholds to make it pass.

## 9. Save the results

In [ ]:
FINAL = os.path.join(DRIVE_DIR, "final")
os.makedirs(FINAL, exist_ok=True)
for net in NETWORKS:
    shutil.copy(os.path.join(MODELS_DIR, f"{net}_gat.pt"), FINAL)
    for pat in (f"gat_eval_{net}.json", f"harp_end_to_end_{net}.json", f"training_{net}_log.csv", f"gat_test_{net}.csv"):
        src = os.path.join(RESULTS_DIR, pat)
        if os.path.exists(src):
            shutil.copy(src, FINAL)
for f in ("experiment_2_summary.csv", "gat_retrain_report.csv"):
    src = os.path.join(RESULTS_DIR, f)
    if os.path.exists(src):
        shutil.copy(src, FINAL)
print("Saved to", FINAL)
print(sorted(os.listdir(FINAL)))

if IN_COLAB and DOWNLOAD_MODELS:
    from google.colab import files
    for net in NETWORKS:
        files.download(os.path.join(FINAL, f"{net}_gat.pt"))

## 10. Put the models back in the project

Copy the downloaded files on your Mac, then re-run the experiments so every number in the slides and README comes from the
new models:

```bash
cp ~/Downloads/Net3_gat.pt ~/Downloads/L-Town_gat.pt backend/data/trained_models/
cp <final>/gat_test_*.csv <final>/experiment_2_summary.csv backend/results/   # optional: refresh GAT accuracy tables
python -m pytest backend/tests -q
```

Experiment 2 reads the generated scenario files and the split, which are not committed. To re-run it locally you need
`python -m backend.gat.dataset_generator --network Net3 --n 3000 --workers 4` first, with the same `--n` as here.

Before presenting, quote the numbers from `gat_retrain_report.csv` (not the older slide figures), and state the scenario counts.